# A Comparative Analysis of IR Models for Book Search Systems

Evaluating Boolean, TF-IDF, and BM25 retrieval models on book data.

### Imports and Get Data

In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from collections import defaultdict, Counter
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

In [3]:
books = pd.read_csv('data/books.csv', low_memory=False)

In [4]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [5]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  object
 1   Book-Title           271360 non-null  object
 2   Book-Author          271358 non-null  object
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  object
 5   Image-URL-S          271360 non-null  object
 6   Image-URL-M          271360 non-null  object
 7   Image-URL-L          271357 non-null  object
dtypes: object(8)
memory usage: 16.6+ MB


### EDA

In [6]:
#Fix column names

books.columns = books.columns.str.replace("-","_").str.replace(" ", "_")

In [7]:
#Remove invalid publication years

before = books.shape[0]

books['Year_Of_Publication'] = pd.to_numeric(books['Year_Of_Publication'], errors='coerce')
books = books[(books['Year_Of_Publication'] >=1800) & (books['Year_Of_Publication']<= 2025)]

after = books.shape[0]

print(f"Rows removed: {before-after}")

Rows removed: 4635


In [8]:
#Normalize book author and book titles

books['Book_Author'] = books['Book_Author'].str.strip().str.title()
books['Book_Title'] = books['Book_Title'].str.strip().str.title()

In [9]:
# Check for blank Authors

books[books['Book_Author'].isna()]

,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
118033,0751352497,A+ Quiz Masters:01 Earth,NaN,1999.0,Dorling Kindersley,http://images.amazon.com/images/P/0751352497.0...,http://images.amazon.com/images/P/0751352497.0...,http://images.amazon.com/images/P/0751352497.0...
187689,9627982032,The Credit Suisse Guide To Managing Your Perso...,NaN,1995.0,Edinburgh Financial Publishing,http://images.amazon.com/images/P/9627982032.0...,http://images.amazon.com/images/P/9627982032.0...,http://images.amazon.com/images/P/9627982032.0...


In [10]:
#Dropping those two rows because they do not matter

books = books.dropna(subset=["Book_Author"])

In [11]:
#Check for blank Publishers

books[books['Publisher'].isna()]

,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
128890,193169656X,Tyrant Moon,Elaine Corvidae,2002.0,NaN,http://images.amazon.com/images/P/193169656X.0...,http://images.amazon.com/images/P/193169656X.0...,http://images.amazon.com/images/P/193169656X.0...
129037,1931696993,Finders Keepers,Linnea Sinclair,2001.0,NaN,http://images.amazon.com/images/P/1931696993.0...,http://images.amazon.com/images/P/1931696993.0...,http://images.amazon.com/images/P/1931696993.0...


In [12]:
#Also dropping those two rows because they do not matter

books = books.dropna(subset=["Publisher"])

In [13]:
#Removing books published by known foreign-language publishers
#Produce books in langauges I do not understand, so excluding them ensures the results remain interpretable

before = books.shape[0]

foreign_publishers = [
    "Goldmann",
    "Heyne",
    "Gallimard",
    "LÃ?Â¼bbe",
    "Distribooks",
    "Eichborn",
    "Diogenes Verlag",
    "Carlsen Verlag GmbH"
]

books = books[
    ~books['Publisher'].str.contains("|".join(foreign_publishers),case = False, na = False)
]

after = books.shape[0]

print(f"Rows removed: {before-after}")

Rows removed: 3970


In [14]:
#Removing Scholastic publisher since that is kids books

kids_books_publishers = [
    "Scholastic", "Random House Books for Young Readers"
]

books = books[
    ~books['Publisher'].str.contains("|".join(kids_books_publishers), case=False, na=False)]

In [15]:
books.head()

,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002.0,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001.0,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision In Normandy,Carlo D'Este,1991.0,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story Of The Great Influenza Pandemic...,Gina Bari Kolata,1999.0,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies Of Urumchi,E. J. W. Barber,1999.0,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [16]:
books = books[
    ['ISBN', 'Book_Title', 'Book_Author', 'Publisher', 'Year_Of_Publication']]

In [17]:
books.head()

,ISBN,Book_Title,Book_Author,Publisher,Year_Of_Publication
0,0195153448,Classical Mythology,Mark P. O. Morford,Oxford University Press,2002.0
1,0002005018,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,2001.0
2,0060973129,Decision In Normandy,Carlo D'Este,HarperPerennial,1991.0
3,0374157065,Flu: The Story Of The Great Influenza Pandemic...,Gina Bari Kolata,Farrar Straus Giroux,1999.0
4,0393045218,The Mummies Of Urumchi,E. J. W. Barber,W. W. Norton &amp; Company,1999.0


### Preprocessing

In [19]:
#Create a searchable text field

books["document"] = (
    books["Book_Title"] + " " +
    books["Book_Author"] + " " +
    books["Publisher"] + " " +
    books["Year_Of_Publication"].astype(str)
)

In [20]:
#Add text preprocessing

def preprocess(text):
    text= text.lower()
    text =re.sub(r"[^a-z0-9\s]", " ", text)
    tokens =text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    return tokens

books["tokens"] = books["document"].apply(preprocess)


In [21]:
#Build inverted index

inverted_index = defaultdict(dict)

for doc_id, tokens in books ["tokens"].items():
    term_counts = Counter(tokens)
    for term, freq in term_counts.items():
        inverted_index[term][doc_id] = freq

### Retrieval Models

In [22]:
#Create Boolean search

def boolean_search (query, top_k=10):
    query_terms = preprocess(query)

    if not query_terms:
        return pd.DataFrame()
    
    matching_docs = set(inverted_index.get(query_terms[0], {}).keys())

    for term in query_terms[1:]:
        matching_docs = matching_docs.intersection(
            set(inverted_index.get(term, {}).keys())
        )

    return books.loc[list(matching_docs),
                    ["Book_Title", "Book_Author", "Publisher", "Year_Of_Publication"]].head(top_k)

In [ ]:
#Optional USER IMPUT
#Test Boolean Search

boolean_search("Mark Rogers")

,Book_Title,Book_Author,Publisher,Year_Of_Publication
200705,The Sword Of Samurai Cat (Tor Fantasy),Mark Rogers,Tor Books,1991.0
187881,The Dead,Mark E. Rogers,Infinity Publishing (PA),2000.0
54247,Samurai Cat Goes To The Movies,Mark E. Rogers,St Martins Pr,1994.0


In [24]:
#Create Vector Space Model with tf-idf

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(books["document"])

def vsm_search(query, top_k=10):
    query_vector = tfidf.transform([query])
    scores = cosine_similarity(query_vector,tfidf_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = books.iloc[top_indices][
        ["Book_Title", "Book_Author", "Publisher", "Year_Of_Publication"].copy()]
    
    results["score"] = scores[top_indices]
    return results

In [ ]:
#Optional USER IMPUT
#Test VSM

vsm_search("romance")

,Book_Title,Book_Author,Publisher,Year_Of_Publication,score
184884,Flight To Romance,Tracy Sinclair,Silhouette Romance,1982.0,0.545086
179511,Romance Inc : Romance Inc,Jocelyn Raines,Pocket,1996.0,0.544572
185229,"To Have, To Hold (Silhouette Romance, #99)",Elaine Camp,Silhouette Romance,1981.0,0.444595
183315,"Ready For Romance (Harlequin Romance, 3288)",Debbie Macomber,Harlequin,1993.0,0.425293
226434,"The Hostage Bride (Silhouette Romance, #82)",Janet Dailey,Silhouette Romance,1981.0,0.421399
34990,"Unlikely Romance (Harlequin Romance, No 3222)",Betty Neels,Harlequin,1992.0,0.421223
168481,"Passion Flower (Silhouette Romance, # 328)",Diana Palmer,Silhouette Romance,1984.0,0.419023
247587,Winter Wedding (Harlequin Romance #2338),Betty Neels,Harlequin Romance,1980.0,0.413858
218104,A Spirited Romance (Zebra Regency Romance),Alana Clayton,Zebra Books,2002.0,0.412445
181526,"Edge Of Paradise (Silhouette Romance, #233)",Dorothy Vernon,Silhouette Romance,1983.0,0.410831


In [26]:
#Add a probabilistic model using BM25

tokenized_corpus = books["tokens"].tolist()
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=10):
    query_tokens = preprocess(query)
    scores = bm25.get_scores(query_tokens)

    top_indices = scores.argsort()[::-1][:top_k]

    results = books.iloc[top_indices][
        ["Book_Title", "Book_Author", "Publisher", "Year_Of_Publication"].copy()]
    
    results["score"]= scores[top_indices]
    return results

In [ ]:
#Optional USER IMPUT
#Test BM25

bm25_search("stephen king")

,Book_Title,Book_Author,Publisher,Year_Of_Publication,score
152999,Stephen King 3,Stephen King,New Amer Library,1988.0,14.278650
261352,Stephen King'S Danse Macabre,Stephen King,Bookthrift Co,1983.0,14.278650
153002,Stephen King (Classics Series),Stephen King,Octopus Books,1987.0,14.278650
236528,Stephen King Danse Macabre,Stephen King,Berkley Publishing Group,1982.0,13.862162
213851,"Stephen King: The Bachman Books, Thinner",Stephen King,Signet Book,1995.0,13.862162
170697,Cujo (The Stephen King Collectors Edition),Stephen King,Penguin USA,1994.0,13.862162
227756,Stephen King'S Danse Macabre,Stephen King,Berkley Publishing Group,2001.0,13.469283
84850,Shining (The Stephen King Collectors Edition),Stephen King,Penguin USA (P),1991.0,13.469283
170719,Bachman Books: Four Early Novels By Stephen King,Stephen King,Bt Bound,2000.0,13.469283
153000,"Stephen King 2: Christine, The Shining, Cujo",Stephen King,Penguin USA,1990.0,13.469283


### Evaluation Set Up

Preparing Queries and Results

In [ ]:
#USER INPUT
#Set evaluation queries

queries = [
    "query term",
    "query term",
    "query term"
]

In [349]:
#Standardize outputs

def run_all_models(query, top_k=10):
    results = []

    b = boolean_search(query, top_k)
    for idx in b.index:
        results.append((query, idx, "Boolean"))

    v = vsm_search(query, top_k)
    for idx in v.index:
        results.append((query, idx, "VSM"))

    m = bm25_search(query, top_k)
    for idx in m.index:
        results.append((query, idx, "BM25"))

    return results

In [350]:
#Combine all model results into one pool
#TREC-style

pool = []

for q in queries:
    pool.extend(run_all_models(q))

pool_df = pd.DataFrame(pool, columns= ["query", "doc_id", "model"])

pool_df = pool_df.merge(
    books[["Book_Title", "Book_Author"]],
    left_on= "doc_id",
    right_index = True,
    how ="left"
)

pool_df = pool_df.drop_duplicates(subset=["query", "doc_id"])

pool_df["title_clean"] = pool_df["Book_Title"].str.lower().str.strip()
pool_df["author_clean"] = pool_df["Book_Author"].str.lower().str.strip()

pool_df = pool_df.drop_duplicates(
    subset=["query", "title_clean", "author_clean"]
).reset_index(drop=True)

pool_df.head(10).sort_values(by="doc_id", ascending =True)

,query,doc_id,model,Book_Title,Book_Author,title_clean,author_clean
9,world war history,77363,Boolean,Pacific Nightmare: How Japan Starts World War ...,Simon Winchester,pacific nightmare: how japan starts world war ...,simon winchester
3,world war history,79639,Boolean,A Short History Of World War Ii,James L. Stokesbury,a short history of world war ii,james l. stokesbury
5,world war history,93470,Boolean,The Good War: An Oral History Of World War Two,Studs Terkel,the good war: an oral history of world war two,studs terkel
7,world war history,121125,Boolean,Naval Battles Of The First World War (Classic ...,Geoffrey Bennett,naval battles of the first world war (classic ...,geoffrey bennett
6,world war history,157346,Boolean,Delivered From Evil : The Saga Of World War Ii...,Robert Leckie,delivered from evil : the saga of world war ii...,robert leckie
0,world war history,176000,Boolean,The American Heritage History Of World War I,American Heritage Editors,the american heritage history of world war i,american heritage editors
2,world war history,241156,Boolean,Return To Diversity: A Political History Of Ea...,Joseph Rothschild,return to diversity: a political history of ea...,joseph rothschild
1,world war history,250884,Boolean,A World At Arms : A Global History Of World Wa...,Gerhard L. Weinberg,a world at arms : a global history of world wa...,gerhard l. weinberg
8,world war history,257072,Boolean,The Second World War: Asia And The Pacific (We...,Thomas E. Griess,the second world war: asia and the pacific (we...,thomas e. griess
4,world war history,269469,Boolean,A History Of Women In The West: Emerging Femin...,Georges Duby,a history of women in the west: emerging femin...,georges duby


Relevance Judgements (Manually Labeled in place of User Interaction Data)

In [351]:
#Add relevance labels

pool_df["relevant"] = 0

In [ ]:
#USER INPUT
#Manually add relevance to each book

for i in range(40):
    row = pool_df.iloc[i]

    print(f"\nIndex: {i}")
    print(f"\Query: {row['query']}")
    print(show_doc(row["doc_id"]))

    label = input("Relevant?") #1 = yes, 0 = no
    pool_df.loc[i, "relevant"] = int(label)


Index: 0
\Query: world war history
Book_Title     The American Heritage History Of World War I
Book_Author                       American Heritage Editors
Name: 176000, dtype: object

Index: 1
\Query: world war history
Book_Title     A World At Arms : A Global History Of World Wa...
Book_Author                                  Gerhard L. Weinberg
Name: 250884, dtype: object

Index: 2
\Query: world war history
Book_Title     Return To Diversity: A Political History Of Ea...
Book_Author                                    Joseph Rothschild
Name: 241156, dtype: object

Index: 3
\Query: world war history
Book_Title     A Short History Of World War Ii
Book_Author                James L. Stokesbury
Name: 79639, dtype: object

Index: 4
\Query: world war history
Book_Title     A History Of Women In The West: Emerging Femin...
Book_Author                                         Georges Duby
Name: 269469, dtype: object


In [ ]:
#Check unlabeled books

pool_df[pool_df["relevant"] == 0]

In [ ]:
#USER INPUT
#Fix unlabeled books

pool_df.loc[[<enter relevant index numbers here>], "relevant"] = 1
pool_df.loc[[<enter NOT relevant index numbers here>], "relevant"] = 2


In [ ]:
#Ensure all results have a relevance label

pool_df.groupby(["model", "query", "relevant"]).size().unstack(fill_value=0)

relevant                       1  2
model   query                      
BM25    stephen king           3  0
        world war history      2  0
Boolean financial management   5  5
        stephen king          10  0
        world war history     10  0
VSM     financial management   4  0
        stephen king           8  0
        world war history      3  3

### Evaluation Metrics

In [362]:
#Precision and Recall

def precision_recall(model_name, query):
    results = run_all_models(query)
    model_docs = [doc for q, doc, m in results if m == model_name]

    rel_docs = pool_df[
        (pool_df["query"] == query) & (pool_df["relevant"]== 1)
    ] ["doc_id"].tolist()

    retrieved_relevant = len(set(model_docs) & set(rel_docs))

    precision = retrieved_relevant / len(model_docs) if model_docs else 0
    recall = retrieved_relevant / len(rel_docs) if rel_docs else 0

    return precision, recall

In [363]:
#nDCG

def ndcg(model_name, query, k=10):
    results = run_all_models(query)
    model_docs = [doc for q, doc, m in results if m== model_name][:k]

    rel_dict = {
        row["doc_id"]: row["relevant"]
        for _, row in pool_df[pool_df["query"] == query].iterrows()
    }
    gains = [rel_dict.get(doc, 0) for doc in model_docs]

    dcg = sum(
        rel/ np.log2(i + 2)
        for i, rel in enumerate(gains)
    )

    ideal = sorted(rel_dict.values(), reverse = True) [:k]
    idcg = sum(
        rel /np.log2(i + 2)
        for i, rel in enumerate(ideal)
    )
    return dcg / idcg if idcg > 0 else 0

In [ ]:
#Retrieval Performance by Model Per-Query

results = []

for q in queries:
    for model in ["Boolean", "VSM", "BM25"]:
        p, r = precision_recall(model, q)
        n = ndcg(model, q)

        results.append({
            "query": q,
            "model": model,
            "precision": p,
            "recall": r,
            "ndcg": n
        })

eval_df = pd.DataFrame(results)
eval_df

,query,model,precision,recall,ndcg
0,world war history,Boolean,1.0,0.666667,0.680735
1,world war history,VSM,0.5,0.333333,0.671940
2,world war history,BM25,0.8,0.533333,0.536265
3,stephen king,Boolean,1.0,0.476190,1.000000
4,stephen king,VSM,0.8,0.380952,0.860382
5,stephen king,BM25,0.9,0.428571,0.861138
6,financial management,Boolean,0.5,0.555556,0.956359
7,financial management,VSM,0.7,0.777778,0.804949
8,financial management,BM25,0.6,0.666667,0.840188


In [ ]:
#Average Retrieval Performance by Model

eval_df.groupby("model")[["precision", "recall", "ndcg"]].mean()

,precision,recall,ndcg
model,,,
BM25,0.766667,0.542857,0.745864
Boolean,0.833333,0.566138,0.879031
VSM,0.666667,0.497354,0.779090
